In [1]:
import sys

sys.path.append("/home/rahim/xelp/work/sqlite-vectordb/")

In [3]:
import pandas as pd
import requests
from src.models import Point, DistanceMetric, CollectionMeta
from src.client import Client
from uuid import uuid4
import numpy as np
from sentence_transformers import SentenceTransformer

In [4]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").to("cuda")

In [5]:
embeddings1 = model.encode("hello")

In [6]:
embeddings2 = model.encode("/home/rahim/xelp/work/qdrant-client/qdrant_local/books.csv")

In [7]:
diff = embeddings1 - embeddings2
print(np.dot(embeddings1, embeddings2))

0.00031258538


In [8]:
df = pd.read_csv("/home/rahim/xelp/work/qdrant-client/qdrant_local/books.csv")

In [9]:
df.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,Unnamed: 12
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.,NaN
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.,NaN
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic,NaN
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.,NaN
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic,NaN


In [10]:
df.drop(labels=["Unnamed: 12"], axis=1, inplace=True)
df.columns

Index(['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13',
       'language_code', '  num_pages', 'ratings_count', 'text_reviews_count',
       'publication_date', 'publisher'],
      dtype='str')

In [6]:
def get_embedding(prompt: str):
    res = requests.post(
        url="http://localhost:8000/v1/embeddings",
        json={
            "model": "sentence-transformers/all-MiniLM-L6-v2",
            "input": "hello",
            "truncate_prompt_tokens": 256,
        },
        timeout=10,
    )
    res.raise_for_status()
    return res.json()["data"][0]["embedding"]

In [7]:
embedding_ = get_embedding("hello")

In [8]:
len(embedding_)

384

In [11]:
client = Client("./test_db")
client.create_collection(
    "books_collection",
    CollectionMeta(
        collection_name="books_collection",
        embedding_size=384,
        distance_metric=DistanceMetric.COSINE,
    ),
)

In [12]:
collection = client.get_collection("books_collection")

In [13]:
columns = df.columns
points = []
for index, row in df.iterrows():
    prompt = ""
    payload = dict()
    for col in columns:
        prompt += col + ": " + str(row[col]) + "\n"
        payload[col] = row[col]
        # embedding = get_embedding(prompt)

    embedding = model.encode(prompt)
    embedding = embedding / np.linalg.norm(embedding)
    point = Point(
        id=str(uuid4()), content=prompt, embedding=embedding.tolist(), payload=payload
    )
    points.append(point)
    if index > 500:
        break

collection.store_points(points=points)

In [14]:
len(collection.ids)

502

In [15]:
query_prompt = "title: Harry Potter\nAuthors: J.K Rowling"
query_embedding = model.encode(query_prompt)

In [16]:
embeddings = np.array(collection.embeddings)
embeddings.shape

(502, 384)

In [19]:
query_embedding = query_embedding / np.linalg.norm(query_embedding)

In [20]:
scores = np.dot(embeddings, query_embedding)

In [21]:
np.dot(embeddings[0], embeddings[4])

np.float64(0.786156011349749)

In [22]:
scores.argmax()

np.int64(6)

In [23]:
collection.ids[6]

'd8a397d3-68c8-4a1d-adae-1f000e132f61'

In [25]:
collection.db.load_point("d8a397d3-68c8-4a1d-adae-1f000e132f61")

Point(id='d8a397d3-68c8-4a1d-adae-1f000e132f61', content='bookID: 10\ntitle: Harry Potter Collection (Harry Potter  #1-6)\nauthors: J.K. Rowling\naverage_rating: 4.73\nisbn: 439827604\nisbn13: 9780439827607\nlanguage_code: eng\n  num_pages: 3342\nratings_count: 28242\ntext_reviews_count: 808\npublication_date: 9/12/2005\npublisher: Scholastic\n', payload={'bookID': 10, 'title': 'Harry Potter Collection (Harry Potter  #1-6)', 'authors': 'J.K. Rowling', 'average_rating': '4.73', 'isbn': '439827604', 'isbn13': 9780439827607, 'language_code': 'eng', '  num_pages': '3342', 'ratings_count': 28242, 'text_reviews_count': 808, 'publication_date': '9/12/2005', 'publisher': 'Scholastic'}, embedding=[-0.04248419031500816, -0.028171909973025322, 0.0024987547658383846, 0.041762858629226685, -0.14663811028003693, 0.04937617853283882, -0.01441285666078329, -0.00431456184014678, 0.014802303165197372, -0.02727227658033371, -0.06952619552612305, 0.011710213497281075, 0.030407432466745377, -0.056607492268

In [20]:
collection.db.close()

# HNSW

In [1]:
"""
HNSW - Hierarchical Navigable Small World Graphs
=================================================
Pure Python + NumPy implementation based on the original paper:
  "Efficient and robust approximate nearest neighbor search using
   Hierarchical Navigable Small World graphs"
   by Yu. A. Malkov, D. A. Yashunin (2018)

Algorithms implemented (matching paper numbering):
  Algorithm 1  - INSERT
  Algorithm 2  - SEARCH-LAYER
  Algorithm 3  - SELECT-NEIGHBORS-SIMPLE
  Algorithm 4  - SELECT-NEIGHBORS-HEURISTIC
  Algorithm 5  - K-NN-SEARCH
"""

'\nHNSW - Hierarchical Navigable Small World Graphs\n=================================================\nPure Python + NumPy implementation based on the original paper:\n  "Efficient and robust approximate nearest neighbor search using\n   Hierarchical Navigable Small World graphs"\n   by Yu. A. Malkov, D. A. Yashunin (2018)\n\nAlgorithms implemented (matching paper numbering):\n  Algorithm 1  - INSERT\n  Algorithm 2  - SEARCH-LAYER\n  Algorithm 3  - SELECT-NEIGHBORS-SIMPLE\n  Algorithm 4  - SELECT-NEIGHBORS-HEURISTIC\n  Algorithm 5  - K-NN-SEARCH\n'

In [2]:
import numpy as np
import math
import heapq
from collections import defaultdict
from typing import List, Set, Dict, Tuple, Optional

In [3]:
def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    """Standard L2 distance between two vectors."""
    diff = a - b
    return float(np.sqrt(np.dot(diff, diff)))


In [4]:
class MinHeap:
    """Min-heap of (distance, element_id) pairs."""

    def __init__(self):
        self._data: List[Tuple[float, int]] = []

    def push(self, dist: float, elem: int):
        heapq.heappush(self._data, (dist, elem))

    def pop(self) -> Tuple[float, int]:
        return heapq.heappop(self._data)

    def peek(self) -> Tuple[float, int]:
        return self._data[0]

    def __len__(self):
        return len(self._data)

    def to_set(self) -> Set[int]:
        return {e for _, e in self._data}

    def to_list(self) -> List[Tuple[float, int]]:
        return list(self._data)


In [5]:
class MaxHeap:
    """Max-heap of (distance, element_id) pairs (negate dist trick)."""

    def __init__(self):
        self._data: List[Tuple[float, int]] = []

    def push(self, dist: float, elem: int):
        heapq.heappush(self._data, (-dist, elem))

    def pop(self) -> Tuple[float, int]:
        neg_d, e = heapq.heappop(self._data)
        return -neg_d, e

    def peek(self) -> Tuple[float, int]:
        neg_d, e = self._data[0]
        return -neg_d, e

    def __len__(self):
        return len(self._data)

    def to_set(self) -> Set[int]:
        return {e for _, e in self._data}

In [6]:
def select_neighbors_simple(
    q: int,
    candidates: Set[int],
    M: int,
    data: np.ndarray,
) -> List[int]:
    """
    Algorithm 3: simply return the M nearest elements from candidates to q.

    Args:
        q          : query element index
        candidates : set of candidate element indices
        M          : number of neighbors to return
        data       : array of all element vectors, shape (N, dim)

    Returns:
        List of up to M nearest element indices from candidates.
    """
    # Sort candidates by distance to q and take the M closest
    dists = [(euclidean_distance(data[q], data[c]), c) for c in candidates]
    dists.sort(key=lambda x: x[0])
    return [e for _, e in dists[:M]]


In [7]:
def select_neighbors_heuristic(
    q: int,
    candidates: Set[int],
    M: int,
    layer: int,
    data: np.ndarray,
    graph: Dict[int, Dict[int, List[int]]],  # graph[layer][node] -> neighbor list
    extend_candidates: bool = True,
    keep_pruned_connections: bool = True,
) -> List[int]:
    """
    Algorithm 4: heuristic neighbor selection that tries to diversify
    the neighborhood - each selected neighbor should be closer to q than
    to any already-selected neighbor.

    Args:
        q                      : query element index
        candidates             : initial set of candidates
        M                      : number of neighbors to return
        layer                  : current graph layer
        data                   : all element vectors
        graph                  : adjacency lists per layer
        extend_candidates      : if True, extend candidates with their neighbors
        keep_pruned_connections: if True, fill up to M with pruned candidates

    Returns:
        List of up to M element indices.
    """
    R: List[int] = []  # result set
    W: List[Tuple[float, int]] = []  # working candidates (min-heap by dist)

    # Build initial working set
    for c in candidates:
        d = euclidean_distance(data[q], data[c])
        heapq.heappush(W, (d, c))

    # (Optional) extend candidates with their layer-lc neighbors
    if extend_candidates:
        extra: Set[int] = set()
        for c in candidates:
            for neighbor in graph.get(layer, {}).get(c, []):
                if neighbor not in candidates:
                    extra.add(neighbor)
        for e_adj in extra:
            d = euclidean_distance(data[q], data[e_adj])
            heapq.heappush(W, (d, e_adj))

    W_d: List[Tuple[float, int]] = []  # discarded candidates

    while W and len(R) < M:
        dist_e, e = heapq.heappop(W)  # nearest in W to q

        # If e is closer to q than to every element already in R → keep it
        closer_to_q = True
        for r in R:
            if euclidean_distance(data[e], data[r]) < dist_e:
                closer_to_q = False
                break

        if closer_to_q:
            R.append(e)
        else:
            W_d.append((dist_e, e))

    # (Optional) fill up remaining slots with pruned connections
    if keep_pruned_connections:
        W_d.sort(key=lambda x: x[0])
        for dist_e, e in W_d:
            if len(R) >= M:
                break
            R.append(e)

    return R

In [8]:
def search_layer(
    q_vec: np.ndarray,
    entry_points: List[int],
    ef: int,
    layer: int,
    data: np.ndarray,
    graph: Dict[int, Dict[int, List[int]]],
) -> List[Tuple[float, int]]:
    """
    Algorithm 2: greedy beam-search on a single graph layer.

    Args:
        q_vec        : query vector (numpy array)
        entry_points : starting element indices
        ef           : size of the dynamic candidate list (beam width)
        layer        : which layer to search
        data         : all element vectors
        graph        : adjacency lists per layer

    Returns:
        List of (distance, element_id) for the ef nearest found neighbors,
        sorted nearest-first.
    """
    v: Set[int] = set(entry_points)  # visited nodes

    # C – candidates (min-heap, nearest at top)
    C: List[Tuple[float, int]] = []
    # W – found nearest neighbors (max-heap, furthest at top so we can prune)
    W: List[Tuple[float, int]] = []  # stored as (-dist, elem)

    for ep in entry_points:
        d = euclidean_distance(q_vec, data[ep])
        heapq.heappush(C, (d, ep))
        heapq.heappush(W, (-d, ep))  # max-heap trick

    while C:
        dist_c, c = heapq.heappop(C)  # nearest candidate to q

        # f = furthest element in W
        neg_dist_f, _ = W[0]
        dist_f = -neg_dist_f

        # If nearest candidate is farther than worst result → stop
        if dist_c > dist_f:
            break

        # Explore neighbors of c on this layer
        for e in graph.get(layer, {}).get(c, []):
            if e not in v:
                v.add(e)
                dist_e = euclidean_distance(q_vec, data[e])

                neg_dist_f2, _ = W[0]
                dist_f2 = -neg_dist_f2

                # Add e to C and W if it improves W or W is not full yet
                if dist_e < dist_f2 or len(W) < ef:
                    heapq.heappush(C, (dist_e, e))
                    heapq.heappush(W, (-dist_e, e))

                    # Keep W trimmed to ef elements
                    if len(W) > ef:
                        heapq.heappop(W)  # removes the furthest (max-heap)

    # Convert W back to (dist, elem) sorted nearest-first
    result = [(-neg_d, e) for neg_d, e in W]
    result.sort(key=lambda x: x[0])
    return result

In [9]:
class HNSW:
    """
    Hierarchical Navigable Small World graph for approximate nearest
    neighbor search.

    Parameters
    ----------
    M              : number of established connections per inserted element
                     (neighbors per layer, except layer 0 which uses M_max0)
    ef_construction: size of the dynamic candidate list during construction
    M_max          : max connections per element per layer  (defaults to M)
    M_max0         : max connections per element at layer 0 (defaults to 2*M)
    m_L            : level normalization factor              (defaults to 1/ln(M))
    use_heuristic  : if True use Algorithm 4, else Algorithm 3
    """

    def __init__(
        self,
        M: int = 16,
        ef_construction: int = 200,
        M_max: Optional[int] = None,
        M_max0: Optional[int] = None,
        m_L: Optional[float] = None,
        use_heuristic: bool = True,
    ):
        self.M = M
        self.ef_construction = ef_construction
        self.M_max = M_max if M_max is not None else M
        self.M_max0 = M_max0 if M_max0 is not None else 2 * M
        self.m_L = m_L if m_L is not None else 1.0 / math.log(M)
        self.use_heuristic = use_heuristic

        # Storage
        self.data: List[np.ndarray] = []  # element vectors
        self.graph: Dict[int, Dict[int, List[int]]] = defaultdict(dict)
        # graph[layer][node] = [neighbors]
        self.enter_point: Optional[int] = None  # global entry point
        self.max_layer: int = -1  # top occupied layer (L)

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _get_level(self) -> int:
        """Sample a random layer level for a new element (paper eq. 1)."""
        # l = floor(-ln(unif(0,1)) * m_L)
        return int(-math.log(np.random.uniform()) * self.m_L)

    def _select_neighbors(
        self,
        q: int,
        candidates: Set[int],
        M: int,
        layer: int,
    ) -> List[int]:
        """Route to Algorithm 3 or 4 depending on self.use_heuristic."""
        if self.use_heuristic:
            return select_neighbors_heuristic(
                q,
                candidates,
                M,
                layer,
                np.array(self.data),
                self.graph,
            )
        else:
            return select_neighbors_simple(
                q,
                candidates,
                M,
                np.array(self.data),
            )

    def _add_connection(self, layer: int, u: int, v: int):
        """Add undirected edge u-v on given layer."""
        if u not in self.graph[layer]:
            self.graph[layer][u] = []
        if v not in self.graph[layer]:
            self.graph[layer][v] = []
        if v not in self.graph[layer][u]:
            self.graph[layer][u].append(v)
        if u not in self.graph[layer][v]:
            self.graph[layer][v].append(u)

    # ------------------------------------------------------------------
    # Algorithm 1 – INSERT
    # ------------------------------------------------------------------

    def insert(self, q_vec: np.ndarray):
        """
        Algorithm 1: insert a new element into the HNSW graph.

        Args:
            q_vec : vector of the new element (numpy array)
        """
        # ---- Bookkeeping ----
        q = len(self.data)  # new element's integer id
        self.data.append(q_vec.copy())
        data_arr = np.array(self.data)  # snapshot for distance calls

        W: List[Tuple[float, int]] = []  # currently found nearest elements
        ep = self.enter_point  # global entry point
        L = self.max_layer  # top layer

        l = self._get_level()  # new element's assigned level

        # ---- Phase 1: greedily descend from L down to l+1 ----
        #      (ef=1: we just track the single nearest neighbor)
        if ep is not None:
            for l_c in range(L, l, -1):
                W = search_layer(
                    q_vec, [ep], ef=1, layer=l_c, data=data_arr, graph=self.graph
                )
                ep = W[0][1]  # nearest element found

            # ---- Phase 2: insert connections from layer min(L,l) down to 0 ----
            for l_c in range(min(L, l), -1, -1):
                W = search_layer(
                    q_vec,
                    [ep],
                    ef=self.ef_construction,
                    layer=l_c,
                    data=data_arr,
                    graph=self.graph,
                )

                candidates = {e for _, e in W}

                M_limit = self.M_max0 if l_c == 0 else self.M_max
                neighbors = self._select_neighbors(q, candidates, self.M, l_c)

                # Add bidirectional connections
                for nb in neighbors:
                    self._add_connection(l_c, q, nb)

                # Shrink connections if any neighbor exceeds M_max
                for nb in neighbors:
                    nb_conn = self.graph[l_c].get(nb, [])
                    if len(nb_conn) > M_limit:
                        nb_candidates = set(nb_conn)
                        new_conn = self._select_neighbors(
                            nb, nb_candidates, M_limit, l_c
                        )
                        self.graph[l_c][nb] = new_conn

                # Move entry point to nearest found in this layer
                ep = W[0][1]

        # ---- Update global entry point if new level is higher ----
        if l > L:
            self.max_layer = l
            self.enter_point = q

            # Ensure graph dicts exist for new layers
            for l_c in range(L + 1, l + 1):
                self.graph[l_c][q] = []

    # ------------------------------------------------------------------
    # Algorithm 5 – K-NN-SEARCH
    # ------------------------------------------------------------------

    def search(
        self, q_vec: np.ndarray, K: int, ef: int = 50
    ) -> List[Tuple[float, int]]:
        """
        Algorithm 5: find the K approximate nearest neighbors of q_vec.

        Args:
            q_vec : query vector
            K     : number of nearest neighbors to return
            ef    : size of dynamic candidate list (≥ K for best recall)

        Returns:
            List of (distance, element_id) sorted nearest-first, length ≤ K.
        """
        if self.enter_point is None:
            return []

        data_arr = np.array(self.data)
        W: List[Tuple[float, int]] = []
        ep = self.enter_point
        L = self.max_layer

        # Greedy descent to layer 1 with ef=1
        for l_c in range(L, 0, -1):
            W = search_layer(
                q_vec, [ep], ef=1, layer=l_c, data=data_arr, graph=self.graph
            )
            ep = W[0][1]

        # Full beam search at layer 0
        W = search_layer(q_vec, [ep], ef=ef, layer=0, data=data_arr, graph=self.graph)

        # Return the K nearest
        W.sort(key=lambda x: x[0])
        return W[:K]

    # ------------------------------------------------------------------
    # Convenience helpers
    # ------------------------------------------------------------------

    def __len__(self) -> int:
        return len(self.data)

    def __repr__(self) -> str:
        return (
            f"HNSW(n={len(self)}, M={self.M}, "
            f"ef_construction={self.ef_construction}, "
            f"layers={self.max_layer + 1})"
        )


In [10]:
if __name__ == "__main__":
    np.random.seed(42)

    DIM = 128  # vector dimension
    N = 1000  # number of elements to index
    K = 10  # nearest neighbors to retrieve
    EF = 32  # search ef

    print("=" * 60)
    print("HNSW Demo")
    print("=" * 60)
    print(f"  Indexing {N} random {DIM}-d vectors …")

    # Build the index
    index = HNSW(M=32, ef_construction=32, use_heuristic=True)
    vectors = np.random.rand(N, DIM).astype(np.float32)
    for i, vec in enumerate(vectors):
        index.insert(vec)
        if (i + 1) % 500 == 0:
            print(f"    inserted {i + 1}/{N}")

    print(f"\n  Index built: {index}")
    print(f"  Layers: 0 … {index.max_layer}")

    # Query
    q_vec = np.random.rand(DIM).astype(np.float32)
    print(f"\n  Querying for {K}-NN …")
    results = index.search(q_vec, K=K, ef=EF)

    print(f"\n  Top-{K} approximate nearest neighbors:")
    for rank, (dist, elem_id) in enumerate(results, 1):
        print(f"    #{rank:2d}  id={elem_id:5d}  dist={dist:.6f}")

    # Brute-force ground truth for recall evaluation
    print("\n  Computing brute-force ground truth …")
    all_dists = np.linalg.norm(vectors - q_vec, axis=1)
    gt_ids = np.argsort(all_dists)[:K]
    gt_set = set(gt_ids.tolist())

    approx_set = {elem_id for _, elem_id in results}
    recall = len(approx_set & gt_set) / K
    print(f"  Recall@{K} = {recall:.2%}")

    print("\n  Brute-force ground truth top-10:")
    for rank, idx in enumerate(gt_ids, 1):
        marker = "✓" if idx in approx_set else "✗"
        print(f"    #{rank:2d}  id={idx:5d}  dist={all_dists[idx]:.6f}  {marker}")

    print("\nDone.")


HNSW Demo
  Indexing 1000 random 128-d vectors …
    inserted 500/1000
    inserted 1000/1000

  Index built: HNSW(n=1000, M=32, ef_construction=32, layers=3)
  Layers: 0 … 2

  Querying for 10-NN …

  Top-10 approximate nearest neighbors:
    # 1  id=  696  dist=3.856275
    # 2  id=  958  dist=3.925118
    # 3  id=  750  dist=3.934429
    # 4  id=  990  dist=3.989977
    # 5  id=  856  dist=4.013902
    # 6  id=  802  dist=4.015751
    # 7  id=  982  dist=4.093794
    # 8  id=  840  dist=4.097702
    # 9  id=   33  dist=4.099151
    #10  id=  736  dist=4.099152

  Computing brute-force ground truth …
  Recall@10 = 90.00%

  Brute-force ground truth top-10:
    # 1  id=  696  dist=3.856275  ✓
    # 2  id=  958  dist=3.925118  ✓
    # 3  id=  750  dist=3.934429  ✓
    # 4  id=  990  dist=3.989977  ✓
    # 5  id=  377  dist=3.992934  ✗
    # 6  id=  856  dist=4.013901  ✓
    # 7  id=  802  dist=4.015751  ✓
    # 8  id=  982  dist=4.093794  ✓
    # 9  id=  840  dist=4.097702  ✓
    #10  